# 01 · Train a model

**AIN 320 final project, Phase 1 (PRD §5.4, §7.2, Checklist Steps 2–5).**

One notebook for all three models: pick `MODEL` and `TIER` below and run all cells.

| Model | Config | Role |
|---|---|---|
| `custom_cnn` | `configs/custom_cnn.yaml` | M1: baseline CNN trained from scratch |
| `resnet50` | `configs/resnet50.yaml` | M2: ImageNet-pretrained, fine-tuned |
| `efficientnet_b0` | `configs/efficientnet_b0.yaml` | M3: ImageNet-pretrained, light and fast |

**Suggested order:** `custom_cnn` on `dev` (minutes, checks everything works), then `main` for each model.
All reported comparisons use the **Main** tier.

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU).

**Disconnected?** Re-run all cells with the same settings. Training resumes from the last finished epoch
(checkpoints are on Drive). To test this on purpose (PRD Step 3), interrupt the training cell after an epoch and
run it again. The log should say `resumed from ...`.

The **test set is not used here**. It is evaluated once per final model in `02_evaluate.ipynb`.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # TODO: your team's GitHub repo
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ain320_doccls"

MODEL = "custom_cnn"   # "custom_cnn" (M1) | "resnet50" (M2) | "efficientnet_b0" (M3)
TIER = "dev"           # "dev" = 3.2k train images, for debugging | "main" = 32k, for reported results
EXTRA = []             # config overrides, e.g. ["train.batch_size=96", "train.epochs=20"]
FRESH = False          # True = ignore this run's checkpoint and train from scratch

## 2 · Mount Drive, get the code, check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone the repo (first run) or pull the latest code, then install dependencies.
import os
REPO_DIR = "/content/final-project"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU (CPU training is far too slow).")

## 3 · Copy the working copy to local disk

Reading thousands of small files from Drive is slow, so the shards built by `00_build_working_copy.ipynb`
are unpacked onto the Colab VM's local disk at the start of every session (a minute or two).

In [ ]:
import os
from pathlib import Path
from src.config import load_config
from src.data.labels import tier_split_lists

WC_DRIVE = f"{DRIVE_ROOT}/working_copy"
WC_LOCAL = "/content/data/working_copy"
!python -m src.data.local_copy unpack --src {WC_DRIVE} --dst {WC_LOCAL}
TRAIN_LIST, VAL_LIST = tier_split_lists(TIER)
!python -m src.data.local_copy verify --data-dir {WC_LOCAL} --lists {TRAIN_LIST} {VAL_LIST}

# Paths for src/ come from the environment: data on local disk, outputs (checkpoints, logs) on Drive.
os.environ["DOCCLS_DATA_DIR"] = WC_LOCAL
os.environ["DOCCLS_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
CONFIG = f"configs/{MODEL}.yaml"
SET_ARGS = " ".join(f"--set {o}" for o in EXTRA)
cfg = load_config(CONFIG, TIER, EXTRA)
RUN_DIR = Path(cfg["paths"]["output_dir"]) / "runs" / cfg["run_name"]
print("config :", CONFIG)
print("run dir:", RUN_DIR)

## 4 · Sanity check: one batch (PRD Step 2)

Shape, dtype, value range and labels of one training batch, plus the augmentations applied to a single page.
Check by eye: small rotations/shifts, brightness, blur and noise only. **No flips.**

In [ ]:
import matplotlib.pyplot as plt
from src.data.dataset import build_datasets, make_loader
from src.data.labels import CLASSES

norm = cfg["data"]["normalize"]
# The custom CNN's real statistics are computed by train.py; any values do for a visual check.
mean, std = (norm["mean"], norm["std"]) if norm["mean"] != "auto" else ([0.5], [0.5])
train_ds, val_ds = build_datasets(cfg, mean, std)
images, labels = next(iter(make_loader(train_ds, 16, shuffle=True, num_workers=0, seed=cfg["seed"])))
print(f"train {len(train_ds)} / val {len(val_ds)} images")
print("batch", tuple(images.shape), images.dtype, f"range [{images.min():.2f}, {images.max():.2f}]")
print("labels", labels.tolist())

def show(tensor):
    """Undo normalization of channel 0 for display."""
    return (tensor[0] * std[0] + mean[0]).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for img, lab, ax in zip(images, labels, axes.flat):
    ax.imshow(show(img), cmap="gray"); ax.set_title(CLASSES[lab], fontsize=9); ax.axis("off")
plt.suptitle("one augmented training batch"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
axes[0].imshow(show(val_ds[0][0]), cmap="gray"); axes[0].set_title("no augmentation")
for ax in axes[1:]:
    ax.imshow(show(train_ds[0][0]), cmap="gray"); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Sanity check: overfit one batch (PRD Step 3)

A correct training loop must memorise 32 images. Expect the loss near 0 and `-> OK` at the end.
If this fails, do not start a long run.

In [ ]:
!python -m src.train --config {CONFIG} --tier {TIER} --overfit-batch 200 {SET_ARGS}

## 6 · Train

Logs one line per 50 batches and one summary line per epoch (validation accuracy, **macro-F1**, invoice-F1).
Early stopping watches validation macro-F1. After each epoch the run folder on Drive holds `last.pt`, `best.pt`,
`history.csv`, `summary.json`, `config.yaml` and `train.log`.

Tip: run Dev first and note `train_minutes` in its summary. Main has 10× the images, so expect about 10× per epoch.

In [ ]:
FRESH_FLAG = "--fresh" if FRESH else ""
!python -m src.train --config {CONFIG} --tier {TIER} {SET_ARGS} {FRESH_FLAG}

## 7 · Results

In [ ]:
import json
import pandas as pd

history = pd.read_csv(RUN_DIR / "history.csv")
PAIR = ["#2a78d6", "#eb6834"]  # blue = first series, orange = second (project chart palette)
style = dict(x="epoch", marker="o", markersize=5, linewidth=2, color=PAIR)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
history.plot(y=["train_loss", "val_loss"], ax=axes[0], title="loss", **style)
history.plot(y=["train_acc", "val_acc"], ax=axes[1], title="accuracy", **style)
history.plot(y=["val_macro_f1", "val_invoice_f1"], ax=axes[2], title="validation F1", **style)
from matplotlib.ticker import MaxNLocator
for ax in axes:
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # whole epochs only
plt.suptitle(f"{cfg['run_name']}"); plt.tight_layout()
plt.savefig(RUN_DIR / "curves.png", dpi=150); plt.show()

summary = json.load(open(RUN_DIR / "summary.json"))
for key in ("stop_reason", "epochs_completed", "best_epoch", "best_val_macro_f1", "best_val_acc",
            "best_val_invoice_f1", "parameters", "best_checkpoint_mb", "train_minutes"):
    print(f"{key:22s} {summary[key]}")
print("GPU:", summary["environment"]["gpu"])

In [ ]:
# Every finished run so far (all models and tiers), for the comparison table.
pd.read_csv(Path(cfg["paths"]["output_dir"]) / "runs_index.csv")

## Next

* Run this notebook again with the other `MODEL` values (Main tier) so all three models use the same data.
* Keep results reproducible: copy `summary.json`, `history.csv`, `config.yaml` and `curves.png` from the run
  folder on Drive into the repo's `results/` folder and commit them. Checkpoints stay on Drive.
* Then evaluate each final model **once** on the test set (`02_evaluate.ipynb`).